# Lecture de OPEN_MEDIC_2025.CSV et analyse R06A


In [1]:
# Si besoin, installer les dependances dans ce notebook:
# %pip install pandas openpyxl

from pathlib import Path
import os
import re
import pandas as pd

# Make paths reproducible whether the notebook is run from the project root or
# from Notebooks/ via nbconvert.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "Data").exists() and (PROJECT_ROOT.parent / "Data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)

SOURCE_DIR = PROJECT_ROOT / "sources" / "open_medic"
REQUIRED_OPEN_MEDIC_FILES = [
    "2025-01-a-06_medic-am-par-classe-atc_serie-mensuelle-labellisee.xlsx",
    "OPEN_PHMEV_2024.CSV",
    "2017-a-2025_retroced-am_serie-annuelle..xlsx",
]
missing_open_medic_files = [
    name for name in REQUIRED_OPEN_MEDIC_FILES
    if not (SOURCE_DIR / name).exists()
]
RUN_OPEN_MEDIC = SOURCE_DIR.exists() and not missing_open_medic_files
SKIP_REASON = (
    "OPEN_MEDIC source files are not available in sources/open_medic: "
    + ", ".join(missing_open_medic_files)
)

print(f"Project root: {PROJECT_ROOT}")
print(f"Source directory: {SOURCE_DIR.relative_to(PROJECT_ROOT)}")
if RUN_OPEN_MEDIC:
    print("OPEN_MEDIC sources found; running full notebook.")
else:
    print("Notebook skipped: " + SKIP_REASON)


Project root: /Users/jo/Desktop/Time Series - Pollen Allergies
Source directory: sources/open_medic
Notebook skipped: OPEN_MEDIC source files are not available in sources/open_medic: 2025-01-a-06_medic-am-par-classe-atc_serie-mensuelle-labellisee.xlsx, OPEN_PHMEV_2024.CSV, 2017-a-2025_retroced-am_serie-annuelle..xlsx


In [2]:
if RUN_OPEN_MEDIC:
    xlsx_path = SOURCE_DIR / "2025-01-a-06_medic-am-par-classe-atc_serie-mensuelle-labellisee.xlsx"
    assert xlsx_path.exists(), f"Fichier introuvable: {xlsx_path}"

    xls = pd.ExcelFile(xlsx_path)
    print("Onglets disponibles:", xls.sheet_names)

else:
    print("Skipped optional OPEN_MEDIC cell: " + SKIP_REASON)


Skipped optional OPEN_MEDIC cell: OPEN_MEDIC source files are not available in sources/open_medic: 2025-01-a-06_medic-am-par-classe-atc_serie-mensuelle-labellisee.xlsx, OPEN_PHMEV_2024.CSV, 2017-a-2025_retroced-am_serie-annuelle..xlsx


## Analyse cible R06A (antihistaminiques systemiques)
Objectif: preparer une base comparable pour Paris (75), Marseille (13), Strasbourg (67), Bordeaux (33).


In [3]:
if RUN_OPEN_MEDIC:
    ville_to_dep = {
        "Paris": "75",
        "Marseille": "13",
        "Strasbourg": "67",
        "Bordeaux": "33",
    }
    target_deps = set(ville_to_dep.values())
    atc_target = "R06A"

    print("Villes cibles:", ville_to_dep)
    print("Departements cibles:", sorted(target_deps))
    print("Code ATC cible:", atc_target)

else:
    print("Skipped optional OPEN_MEDIC cell: " + SKIP_REASON)


Skipped optional OPEN_MEDIC cell: OPEN_MEDIC source files are not available in sources/open_medic: 2025-01-a-06_medic-am-par-classe-atc_serie-mensuelle-labellisee.xlsx, OPEN_PHMEV_2024.CSV, 2017-a-2025_retroced-am_serie-annuelle..xlsx


In [4]:
if RUN_OPEN_MEDIC:
    sheet_name = "2025_atc3_total"
    df_atc3 = pd.read_excel(xlsx_path, sheet_name=sheet_name, header=5)
    df_atc3.columns = (
        df_atc3.columns.astype(str)
        .str.replace("\n", " ", regex=False)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    code_col = next(c for c in df_atc3.columns if c.lower().startswith("code atc3"))
    lib_col = next(c for c in df_atc3.columns if ("atc3" in c.lower() and c != code_col))

    print("Onglet charge:", sheet_name)
    print("Shape:", df_atc3.shape)
    print("Colonnes code/libelle:", code_col, "|", lib_col)
    df_atc3.head(3)

else:
    print("Skipped optional OPEN_MEDIC cell: " + SKIP_REASON)


Skipped optional OPEN_MEDIC cell: OPEN_MEDIC source files are not available in sources/open_medic: 2025-01-a-06_medic-am-par-classe-atc_serie-mensuelle-labellisee.xlsx, OPEN_PHMEV_2024.CSV, 2017-a-2025_retroced-am_serie-annuelle..xlsx


In [5]:
if RUN_OPEN_MEDIC:
    import re

else:
    print("Skipped optional OPEN_MEDIC cell: " + SKIP_REASON)


Skipped optional OPEN_MEDIC cell: OPEN_MEDIC source files are not available in sources/open_medic: 2025-01-a-06_medic-am-par-classe-atc_serie-mensuelle-labellisee.xlsx, OPEN_PHMEV_2024.CSV, 2017-a-2025_retroced-am_serie-annuelle..xlsx


In [6]:
if RUN_OPEN_MEDIC:
    monthly_cols = [c for c in df_atc3.columns if re.search(r"20\d{2}-\d{2}$", c)]
    months = sorted({re.search(r"(20\d{2}-\d{2})$", c).group(1) for c in monthly_cols})
    metrics = sorted({re.sub(r"\s*20\d{2}-\d{2}$", "", c).strip() for c in monthly_cols})

    print("Aggregation temporelle actuelle:")
    print("- granularite: mensuelle")
    print("- fenetre: ", months[0], "a", months[-1])
    print("- nb mois: ", len(months))
    print("- indicateurs mensuels: ", metrics)

else:
    print("Skipped optional OPEN_MEDIC cell: " + SKIP_REASON)


Skipped optional OPEN_MEDIC cell: OPEN_MEDIC source files are not available in sources/open_medic: 2025-01-a-06_medic-am-par-classe-atc_serie-mensuelle-labellisee.xlsx, OPEN_PHMEV_2024.CSV, 2017-a-2025_retroced-am_serie-annuelle..xlsx


In [7]:
if RUN_OPEN_MEDIC:
    df_r06a = df_atc3.loc[df_atc3[code_col].eq(atc_target)].copy()
    assert not df_r06a.empty, "R06A introuvable dans cet onglet"

    value_cols = [c for c in df_r06a.columns if re.search(r"20\d{2}-\d{2}$", c)]
    df_r06a_long = df_r06a.melt(
        id_vars=[code_col, lib_col],
        value_vars=value_cols,
        var_name="variable_mois",
        value_name="valeur",
    )
    df_r06a_long["mois"] = df_r06a_long["variable_mois"].str.extract(r"(20\d{2}-\d{2})$")
    df_r06a_long["indicateur"] = (
        df_r06a_long["variable_mois"]
        .str.replace(r"\s*20\d{2}-\d{2}$", "", regex=True)
        .str.strip()
    )
    df_r06a_long = df_r06a_long[[code_col, lib_col, "mois", "indicateur", "valeur"]]
    df_r06a_long = df_r06a_long.sort_values(["mois", "indicateur"]).reset_index(drop=True)

    print("R06A detecte:", df_r06a.iloc[0][lib_col])
    df_r06a_long.head(12)

else:
    print("Skipped optional OPEN_MEDIC cell: " + SKIP_REASON)


Skipped optional OPEN_MEDIC cell: OPEN_MEDIC source files are not available in sources/open_medic: 2025-01-a-06_medic-am-par-classe-atc_serie-mensuelle-labellisee.xlsx, OPEN_PHMEV_2024.CSV, 2017-a-2025_retroced-am_serie-annuelle..xlsx


In [8]:
if RUN_OPEN_MEDIC:
    # Etape 3 (taux pour 1000 habitants) - pre-requis population departementale
    # BEN_REG est un code region: on mappe les departements 13/33/67/75 vers leurs regions.
    pop_dep = pd.Series(
        {"13": 2087658, "33": 1690493, "67": 1163810, "75": 2103778},
        name="population",
    ).rename_axis("dep").reset_index()

    dep_to_ben_reg = {
        "13": "93",  # Provence-Alpes-Cote d'Azur
        "33": "75",  # Nouvelle-Aquitaine
        "67": "44",  # Grand Est
        "75": "11",  # Ile-de-France
    }
    pop_dep["BEN_REG"] = pop_dep["dep"].map(dep_to_ben_reg)
    display(pop_dep.sort_values("dep").reset_index(drop=True))

    if "df" in locals() and {"BEN_REG", "ATC3", "BOITES", "REM"}.issubset(df.columns):
        target_ben_reg = sorted(pop_dep["BEN_REG"].dropna().unique().tolist())
        df["BEN_REG_STR"] = df["BEN_REG"].astype(str).str.strip().str.zfill(2)
        df_scope = df.loc[df["BEN_REG_STR"].isin(target_ben_reg)].copy()

        print("Codes BEN_REG cibles:", target_ben_reg)
        print("Shape filtree (BEN_REG):", df_scope.shape)
        print("BEN_REG presents apres filtre:", sorted(df_scope["BEN_REG_STR"].unique()))

        resume_ben_reg = (
            df_scope.groupby("BEN_REG_STR", as_index=False)
            .agg(nb_lignes=("ATC3", "size"), boites=("BOITES", "sum"), rem=("REM", "sum"))
            .sort_values("BEN_REG_STR")
        )
        resume_ben_reg = resume_ben_reg.merge(
            pop_dep[["dep", "population", "BEN_REG"]],
            left_on="BEN_REG_STR",
            right_on="BEN_REG",
            how="left",
        )
        resume_ben_reg["boites_pour_1000_hab"] = resume_ben_reg["boites"] / resume_ben_reg["population"] * 1000
        display(resume_ben_reg[["dep", "BEN_REG_STR", "population", "nb_lignes", "boites", "rem", "boites_pour_1000_hab"]])
    else:
        print("INFO: la base OPEN_MEDIC detaillee (df) n est pas chargee dans ce notebook.")
        print("Le mapping dep->BEN_REG est conserve et sera reutilise plus bas pour la cible mensuelle.")

    print("Note: avec BEN_REG, les volumes sont regionaux (pas strictement departementaux).")

else:
    print("Skipped optional OPEN_MEDIC cell: " + SKIP_REASON)


Skipped optional OPEN_MEDIC cell: OPEN_MEDIC source files are not available in sources/open_medic: 2025-01-a-06_medic-am-par-classe-atc_serie-mensuelle-labellisee.xlsx, OPEN_PHMEV_2024.CSV, 2017-a-2025_retroced-am_serie-annuelle..xlsx


In [9]:
if RUN_OPEN_MEDIC:
    # Etape 4 - Inventaire des fichiers OPEN MEDIC (xlsx) et controle de completude annuelle
    import re
    from pathlib import Path

    exclude_file = "2024_descriptif-variables_open-medic.xls"
    pattern = re.compile(
        r"^(?P<annee>\d{4})-(?P<semestre>01-a-06|07-a-12)_medic-am-par-classe-atc_serie-mensuelle(?:-labellisee)?\.(?P<ext>xlsx|zip)$"
    )

    catalog = []
    for p in sorted(SOURCE_DIR.iterdir()):
        if not p.is_file() or p.name == exclude_file:
            continue
        m = pattern.match(p.name)
        if not m:
            continue
        catalog.append({
            "fichier": p.name,
            "annee": int(m.group("annee")),
            "semestre": m.group("semestre"),
            "ext": m.group("ext"),
            "path": str(p),
        })

    inventory_df = pd.DataFrame(catalog).sort_values(["annee", "semestre", "ext", "fichier"]).reset_index(drop=True)
    display(inventory_df)

    expected_semesters = {"01-a-06", "07-a-12"}
    years = sorted(inventory_df["annee"].unique().tolist()) if not inventory_df.empty else []

    alerts = []
    for annee in years:
        y = inventory_df.loc[inventory_df["annee"].eq(annee)]
        xlsx_sem = set(y.loc[y["ext"].eq("xlsx"), "semestre"].tolist())
        zip_sem = set(y.loc[y["ext"].eq("zip"), "semestre"].tolist())
        missing_xlsx = sorted(expected_semesters - xlsx_sem)
        if missing_xlsx:
            alerts.append(
                f"Annee {annee}: xlsx manquant(s) pour {missing_xlsx}. Semestres zip presents: {sorted(zip_sem) if zip_sem else []}"
            )

    if alerts:
        print("ALERTE COMPLETUDE (xlsx):")
        for msg in alerts:
            print("-", msg)
    else:
        print("OK: chaque annee detectee dispose des 2 fichiers xlsx (01-a-06 et 07-a-12).")

else:
    print("Skipped optional OPEN_MEDIC cell: " + SKIP_REASON)


Skipped optional OPEN_MEDIC cell: OPEN_MEDIC source files are not available in sources/open_medic: 2025-01-a-06_medic-am-par-classe-atc_serie-mensuelle-labellisee.xlsx, OPEN_PHMEV_2024.CSV, 2017-a-2025_retroced-am_serie-annuelle..xlsx


In [10]:
if RUN_OPEN_MEDIC:
    # Etape 5 - Chargement de tous les xlsx et mise au format long mensuel
    def _clean_columns(df_in: pd.DataFrame) -> pd.DataFrame:
        df = df_in.copy()
        df.columns = (
            df.columns.astype(str)
            .str.replace("\n", " ", regex=False)
            .str.replace(r"\s+", " ", regex=True)
            .str.strip()
        )
        return df

    def _pick_atc3_sheet(xls: pd.ExcelFile, annee: int) -> str:
        exact = f"{annee}_atc3_total"
        if exact in xls.sheet_names:
            return exact
        candidates = [s for s in xls.sheet_names if "atc3_total" in s.lower()]
        if len(candidates) == 1:
            return candidates[0]
        year_candidates = [s for s in candidates if str(annee) in s]
        if len(year_candidates) == 1:
            return year_candidates[0]
        raise ValueError(f"Impossible de choisir un onglet atc3_total pour {annee}. Onglets: {xls.sheet_names}")

    xlsx_inventory = inventory_df.loc[inventory_df["ext"].eq("xlsx")].copy()
    frames = []
    load_errors = []

    for _, row in xlsx_inventory.iterrows():
        annee = int(row["annee"])
        semestre = row["semestre"]
        file_path = Path(row["path"])
        try:
            xls = pd.ExcelFile(file_path)
            sheet_name = _pick_atc3_sheet(xls, annee)
            df_atc3 = pd.read_excel(file_path, sheet_name=sheet_name, header=5)
            df_atc3 = _clean_columns(df_atc3)

            code_col = next(c for c in df_atc3.columns if c.lower().startswith("code atc3"))
            lib_col = next(c for c in df_atc3.columns if ("atc3" in c.lower() and c != code_col))
            value_cols = [c for c in df_atc3.columns if re.search(r"20\d{2}-\d{2}$", c)]

            df_long = df_atc3.melt(
                id_vars=[code_col, lib_col],
                value_vars=value_cols,
                var_name="variable_mois",
                value_name="valeur",
            )
            df_long["mois"] = df_long["variable_mois"].str.extract(r"(20\d{2}-\d{2})$")
            df_long["indicateur"] = (
                df_long["variable_mois"]
                .str.replace(r"\s*20\d{2}-\d{2}$", "", regex=True)
                .str.strip()
            )
            df_long = df_long.rename(columns={code_col: "code_atc3", lib_col: "libelle_atc3"})
            df_long["annee"] = annee
            df_long["semestre"] = semestre
            df_long["fichier_source"] = file_path.name
            df_long = df_long[[
                "annee", "semestre", "mois", "code_atc3", "libelle_atc3",
                "indicateur", "valeur", "fichier_source"
            ]]
            frames.append(df_long)
        except Exception as e:
            load_errors.append({
                "fichier": file_path.name,
                "annee": annee,
                "semestre": semestre,
                "erreur": str(e),
            })

    if load_errors:
        print("ERREURS DE CHARGEMENT:")
        display(pd.DataFrame(load_errors))

    if frames:
        df_atc3_all = pd.concat(frames, ignore_index=True)
        df_atc3_all = df_atc3_all.sort_values(["mois", "code_atc3", "indicateur"]).reset_index(drop=True)
        print("Dataset fusionne cree: df_atc3_all")
        print("Shape:", df_atc3_all.shape)
        print("Periode:", df_atc3_all["mois"].min(), "a", df_atc3_all["mois"].max())
        display(df_atc3_all.head(10))
    else:
        print("Aucun xlsx charge. Verifie l inventaire et les erreurs ci-dessus.")

else:
    print("Skipped optional OPEN_MEDIC cell: " + SKIP_REASON)


Skipped optional OPEN_MEDIC cell: OPEN_MEDIC source files are not available in sources/open_medic: 2025-01-a-06_medic-am-par-classe-atc_serie-mensuelle-labellisee.xlsx, OPEN_PHMEV_2024.CSV, 2017-a-2025_retroced-am_serie-annuelle..xlsx


In [11]:
if RUN_OPEN_MEDIC:
    # Etape 6 - Merge final et notification de completude par annee
    if "df_atc3_all" not in locals():
        raise ValueError("df_atc3_all est absent. Execute l Etape 5 avant cette cellule.")

    expected_by_sem = {"01-a-06": {f"{m:02d}" for m in range(1, 7)}, "07-a-12": {f"{m:02d}" for m in range(7, 13)}}
    check_rows = []

    for annee in sorted(inventory_df["annee"].unique().tolist()):
        y = inventory_df.loc[inventory_df["annee"].eq(annee)]
        xlsx_sem = sorted(y.loc[y["ext"].eq("xlsx"), "semestre"].unique().tolist())
        expected_months = sorted({m for sem in xlsx_sem for m in expected_by_sem.get(sem, set())})

        observed_months = sorted(
            df_atc3_all.loc[df_atc3_all["annee"].eq(annee), "mois"]
            .dropna()
            .str[-2:]
            .unique()
            .tolist()
        )
        missing_months = sorted(set(expected_months) - set(observed_months))
        missing_sem_xlsx = sorted(set(["01-a-06", "07-a-12"]) - set(xlsx_sem))

        check_rows.append({
            "annee": annee,
            "xlsx_semestres_presents": ", ".join(xlsx_sem) if xlsx_sem else "",
            "xlsx_semestres_manquants": ", ".join(missing_sem_xlsx) if missing_sem_xlsx else "",
            "mois_observes": ", ".join(observed_months),
            "mois_attendus_depuis_xlsx": ", ".join(expected_months),
            "mois_manquants_dans_merge": ", ".join(missing_months) if missing_months else "",
            "status": "OK" if not missing_sem_xlsx and not missing_months else "ALERTE",
        })

    check_df = pd.DataFrame(check_rows).sort_values("annee").reset_index(drop=True)
    display(check_df)

    alerts = check_df.loc[check_df["status"].eq("ALERTE")]
    if alerts.empty:
        print("OK: merge complet et 2 fichiers xlsx presents pour chaque annee detectee.")
    else:
        print("ALERTE: des semestres ou des mois manquent. Voir check_df ci-dessus.")

else:
    print("Skipped optional OPEN_MEDIC cell: " + SKIP_REASON)


Skipped optional OPEN_MEDIC cell: OPEN_MEDIC source files are not available in sources/open_medic: 2025-01-a-06_medic-am-par-classe-atc_serie-mensuelle-labellisee.xlsx, OPEN_PHMEV_2024.CSV, 2017-a-2025_retroced-am_serie-annuelle..xlsx


In [12]:
if RUN_OPEN_MEDIC:
    # Etape 2 — Passer du daily au mensuel (Option A) pour un merge compatible avec Ameli
    pollen_daily_path = Path("Data/processed/pollen_weather_merged.csv")
    hpi_daily_path = Path("Data/processed/hpi_daily.csv")
    assert pollen_daily_path.exists(), f"Fichier introuvable: {pollen_daily_path}"
    assert hpi_daily_path.exists(), f"Fichier introuvable: {hpi_daily_path}"

    df_pollen_daily = pd.read_csv(pollen_daily_path, parse_dates=["date"])
    df_hpi_daily = pd.read_csv(hpi_daily_path, parse_dates=["date"])

    pollen_species_cols = [
        "birch_pollen", "alder_pollen", "grass_pollen",
        "olive_pollen", "mugwort_pollen", "ragweed_pollen",
    ]
    missing_cols = [c for c in pollen_species_cols if c not in df_pollen_daily.columns]
    assert not missing_cols, f"Colonnes pollen manquantes: {missing_cols}"
    assert "HPI" in df_hpi_daily.columns, "Colonne HPI absente de hpi_daily.csv"

    df_pollen_daily["pollen_total"] = df_pollen_daily[pollen_species_cols].sum(axis=1, min_count=1)
    df_daily_city = df_pollen_daily[["date", "city", "pollen_total"]].merge(
        df_hpi_daily[["date", "city", "HPI"]],
        on=["date", "city"],
        how="inner",
        validate="one_to_one",
    )
    df_daily_city["mois"] = df_daily_city["date"].dt.to_period("M").astype(str)

    df_city_monthly = (
        df_daily_city.groupby(["city", "mois"], as_index=False)
        .agg(
            pollen_moyen=("pollen_total", "mean"),
            hpi_moyen=("HPI", "mean"),
            nb_jours=("date", "nunique"),
        )
        .sort_values(["city", "mois"])
        .reset_index(drop=True)
    )

    print("Base mensuelle city-level (pollen + HPI):", df_city_monthly.shape)
    print("Periode:", df_city_monthly["mois"].min(), "a", df_city_monthly["mois"].max())
    display(df_city_monthly.head(12))

else:
    print("Skipped optional OPEN_MEDIC cell: " + SKIP_REASON)


Skipped optional OPEN_MEDIC cell: OPEN_MEDIC source files are not available in sources/open_medic: 2025-01-a-06_medic-am-par-classe-atc_serie-mensuelle-labellisee.xlsx, OPEN_PHMEV_2024.CSV, 2017-a-2025_retroced-am_serie-annuelle..xlsx


In [13]:
if RUN_OPEN_MEDIC:
    # Etape 3 — Construire la variable cible (taux de remboursement pour 1000 habitants)
    import unicodedata

    if "df_atc3_all" not in locals():
        raise ValueError("df_atc3_all est absent. Execute d abord les cellules Etape 4-5-6 de la partie Ameli.")

    city_to_dep = {
        "Paris": "75",
        "Marseille": "13",
        "Strasbourg": "67",
        "Bordeaux": "33",
    }
    dep_to_ben_reg = {"13": "93", "33": "75", "67": "44", "75": "11"}
    dep_population = {"13": 2087658, "33": 1690493, "67": 1163810, "75": 2103778}

    city_meta = (
        pd.DataFrame({"city": list(city_to_dep.keys()), "dep": list(city_to_dep.values())})
        .assign(
            BEN_REG=lambda d: d["dep"].map(dep_to_ben_reg),
            population=lambda d: d["dep"].map(dep_population),
        )
    )

    def _txt_norm(x: str) -> str:
        s = unicodedata.normalize("NFKD", str(x)).encode("ascii", "ignore").decode("ascii")
        return re.sub(r"\s+", " ", s).strip().lower()

    df_r06a = df_atc3_all.loc[df_atc3_all["code_atc3"].eq("R06A")].copy()
    if df_r06a.empty:
        raise ValueError("R06A absent de df_atc3_all")

    ind_candidates = [
        ind for ind in df_r06a["indicateur"].dropna().unique().tolist()
        if ("boite" in _txt_norm(ind)) and ("rembours" in _txt_norm(ind))
    ]
    if not ind_candidates:
        raise ValueError("Indicateur boites remboursées introuvable pour R06A")

    target_indicator = sorted(ind_candidates)[0]
    print("Indicateur cible retenu:", target_indicator)

    df_target_fr = (
        df_r06a.loc[df_r06a["indicateur"].eq(target_indicator), ["mois", "valeur"]]
        .assign(boites_r06a_fr=lambda d: pd.to_numeric(d["valeur"], errors="coerce"))
        [["mois", "boites_r06a_fr"]]
        .dropna(subset=["boites_r06a_fr"])
        .groupby("mois", as_index=False)["boites_r06a_fr"].sum()
        .sort_values("mois")
        .reset_index(drop=True)
    )

    # Allocation ville basée uniquement sur la population des 4 départements.
    city_weights = city_meta.copy()
    city_weights["poids_alloc"] = city_weights["population"] / city_weights["population"].sum()
    allocation_mode = "poids populationnels (base: df_atc3_all + populations dep)"

    print("Mode d allocation:", allocation_mode)
    display(city_weights)

    df_target_city = city_weights.assign(_k=1).merge(df_target_fr.assign(_k=1), on="_k").drop(columns="_k")
    df_target_city["boites_r06a_city_est"] = df_target_city["boites_r06a_fr"] * df_target_city["poids_alloc"]
    df_target_city["remboursement_p1000"] = df_target_city["boites_r06a_city_est"] / df_target_city["population"] * 1000

    df_model_monthly = df_city_monthly.merge(
        df_target_city[["city", "mois", "dep", "BEN_REG", "population", "boites_r06a_city_est", "remboursement_p1000"]],
        on=["city", "mois"],
        how="inner",
    )
    df_model_monthly = df_model_monthly.sort_values(["city", "mois"]).reset_index(drop=True)

    print("Base modele mensuelle creee:", df_model_monthly.shape)
    print("Periode commune pollen/HPI/Ameli:", df_model_monthly["mois"].min(), "a", df_model_monthly["mois"].max())
    display(df_model_monthly.head(12))

else:
    print("Skipped optional OPEN_MEDIC cell: " + SKIP_REASON)


Skipped optional OPEN_MEDIC cell: OPEN_MEDIC source files are not available in sources/open_medic: 2025-01-a-06_medic-am-par-classe-atc_serie-mensuelle-labellisee.xlsx, OPEN_PHMEV_2024.CSV, 2017-a-2025_retroced-am_serie-annuelle..xlsx


In [14]:
if RUN_OPEN_MEDIC:
    # Etape 4 — Test central: Modele A (Pollen) vs Modele B (Pollen + HPI)
    import statsmodels.api as sm

    if "df_model_monthly" not in locals():
        raise ValueError("df_model_monthly est absent. Execute d abord Etape 2 et Etape 3.")

    df_global = (
        df_model_monthly.groupby("mois", as_index=False)
        .agg(
            remboursement_p1000=("remboursement_p1000", "mean"),
            pollen_moyen=("pollen_moyen", "mean"),
            hpi_moyen=("hpi_moyen", "mean"),
            n_villes=("city", "nunique"),
        )
        .dropna()
    )

    X_a = sm.add_constant(df_global[["pollen_moyen"]], has_constant="add")
    X_b = sm.add_constant(df_global[["pollen_moyen", "hpi_moyen"]], has_constant="add")
    y = df_global["remboursement_p1000"]

    model_a = sm.OLS(y, X_a).fit()
    model_b = sm.OLS(y, X_b).fit()
    f_stat, p_val, df_diff = model_b.compare_f_test(model_a)

    comparison_global = pd.DataFrame([
        {"modele": "A (pollen)", "R2": model_a.rsquared, "R2_adj": model_a.rsquared_adj, "AIC": model_a.aic, "BIC": model_a.bic},
        {"modele": "B (pollen + HPI)", "R2": model_b.rsquared, "R2_adj": model_b.rsquared_adj, "AIC": model_b.aic, "BIC": model_b.bic},
    ])
    comparison_global["delta_R2_vs_A"] = comparison_global["R2"] - comparison_global.loc[0, "R2"]
    display(comparison_global)

    print("Nested F-test (B vs A):")
    print(f"- F = {f_stat:.4f}")
    print(f"- p-value = {p_val:.6f}")
    print(f"- ddl diff = {int(df_diff)}")
    print("Interpretation: p < 0.05 suggere que l ajout de HPI ameliore significativement le modele.")

else:
    print("Skipped optional OPEN_MEDIC cell: " + SKIP_REASON)


Skipped optional OPEN_MEDIC cell: OPEN_MEDIC source files are not available in sources/open_medic: 2025-01-a-06_medic-am-par-classe-atc_serie-mensuelle-labellisee.xlsx, OPEN_PHMEV_2024.CSV, 2017-a-2025_retroced-am_serie-annuelle..xlsx


In [15]:
if RUN_OPEN_MEDIC:
    # Etape 5 — Comparer le gain du modele B par ville
    if "df_model_monthly" not in locals():
        raise ValueError("df_model_monthly est absent. Execute d abord Etape 2 et Etape 3.")

    rows = []
    for city, g in df_model_monthly.groupby("city"):
        g = g.dropna(subset=["remboursement_p1000", "pollen_moyen", "hpi_moyen"]).copy()
        if g.shape[0] < 8:
            rows.append({
                "city": city,
                "n_obs": g.shape[0],
                "R2_A": float("nan"),
                "R2_B": float("nan"),
                "delta_R2": float("nan"),
                "p_value_Ftest": float("nan"),
                "status": "trop peu d observations",
            })
            continue

        X_a = sm.add_constant(g[["pollen_moyen"]], has_constant="add")
        X_b = sm.add_constant(g[["pollen_moyen", "hpi_moyen"]], has_constant="add")
        y = g["remboursement_p1000"]

        m_a = sm.OLS(y, X_a).fit()
        m_b = sm.OLS(y, X_b).fit()
        f_stat, p_val, df_diff = m_b.compare_f_test(m_a)

        rows.append({
            "city": city,
            "n_obs": int(g.shape[0]),
            "R2_A": m_a.rsquared,
            "R2_B": m_b.rsquared,
            "R2_adj_A": m_a.rsquared_adj,
            "R2_adj_B": m_b.rsquared_adj,
            "delta_R2": m_b.rsquared - m_a.rsquared,
            "F_stat": f_stat,
            "p_value_Ftest": p_val,
            "status": "OK",
        })

    city_model_comparison = pd.DataFrame(rows).sort_values("delta_R2", ascending=False).reset_index(drop=True)
    display(city_model_comparison)

    if "allocation_mode" in locals():
        print("Note allocation:", allocation_mode)
        print("Pour une inference forte par ville, il faut idealement des remboursements mensuels territoriaux observes (region/departement).")

else:
    print("Skipped optional OPEN_MEDIC cell: " + SKIP_REASON)


Skipped optional OPEN_MEDIC cell: OPEN_MEDIC source files are not available in sources/open_medic: 2025-01-a-06_medic-am-par-classe-atc_serie-mensuelle-labellisee.xlsx, OPEN_PHMEV_2024.CSV, 2017-a-2025_retroced-am_serie-annuelle..xlsx


In [16]:
if RUN_OPEN_MEDIC:
    # Etape 7 — Etendre le dataset Ameli mensuel avec les fichiers 2014
    def _infer_year_sem_from_name(name: str):
        m_std = re.search(
            r"(?P<annee>\d{4})-(?P<semestre>01-a-06|07-a-12)_medic-am-par-classe-atc_serie-mensuelle(?:-labellisee)?\.xlsx$",
            name,
            flags=re.IGNORECASE,
        )
        if m_std:
            return int(m_std.group("annee")), m_std.group("semestre")

        m_2014_s1 = re.search(r"2014_1er\s+semestre", name, flags=re.IGNORECASE)
        if m_2014_s1:
            return 2014, "01-a-06"

        m_2014_s2 = re.search(r"2014_2eme\s+semestre", name, flags=re.IGNORECASE)
        if m_2014_s2:
            return 2014, "07-a-12"

        return None

    def _read_atc3_total_any_header(file_path: Path, annee: int):
        xls = pd.ExcelFile(file_path)
        sheet_name = _pick_atc3_sheet(xls, annee)
        # Les fichiers changent parfois de ligne d en-tête (4 ou 5 en index 0-based).
        trials = []
        for hdr in [5, 4]:
            try:
                df_try = pd.read_excel(file_path, sheet_name=sheet_name, header=hdr)
                df_try = _clean_columns(df_try)
                code_col = next(c for c in df_try.columns if str(c).lower().startswith("code atc3"))
                lib_col = next(c for c in df_try.columns if ("atc3" in str(c).lower() and c != code_col))
                return df_try, code_col, lib_col, sheet_name, hdr
            except Exception as e:
                trials.append((hdr, str(e)))
        raise ValueError(f"Lecture impossible pour {file_path.name} / {sheet_name}. Essais: {trials}")

    all_xlsx = sorted(SOURCE_DIR.glob("*.xlsx"))
    catalog = []
    for p in all_xlsx:
        if p.name == "2017-a-2025_retroced-am_serie-annuelle..xlsx":
            continue
        parsed = _infer_year_sem_from_name(p.name)
        if parsed is None:
            continue
        annee, semestre = parsed
        catalog.append({"fichier": p.name, "path": str(p), "annee": annee, "semestre": semestre})

    inventory_extended = pd.DataFrame(catalog).sort_values(["annee", "semestre", "fichier"]).reset_index(drop=True)
    display(inventory_extended)

    expected_semesters = {"01-a-06", "07-a-12"}
    missing_msgs = []
    for annee in sorted(inventory_extended["annee"].unique().tolist()):
        sems = set(inventory_extended.loc[inventory_extended["annee"].eq(annee), "semestre"].tolist())
        miss = sorted(expected_semesters - sems)
        if miss:
            missing_msgs.append(f"Annee {annee}: semestre(s) manquant(s) {miss}")

    if missing_msgs:
        print("ALERTE COMPLETUDE (xlsx) :")
        for msg in missing_msgs:
            print("-", msg)
    else:
        print("OK: chaque annee detectee dispose des 2 semestres xlsx.")

    frames = []
    load_errors = []
    for _, row in inventory_extended.iterrows():
        file_path = Path(row["path"])
        annee = int(row["annee"])
        semestre = row["semestre"]
        try:
            df_atc3, code_col, lib_col, sheet_name, hdr = _read_atc3_total_any_header(file_path, annee)
            value_cols = [c for c in df_atc3.columns if re.search(r"20\d{2}-\d{2}$", str(c))]
            df_long = df_atc3.melt(
                id_vars=[code_col, lib_col],
                value_vars=value_cols,
                var_name="variable_mois",
                value_name="valeur",
            )
            df_long["mois"] = df_long["variable_mois"].astype(str).str.extract(r"(20\d{2}-\d{2})$")
            df_long["indicateur"] = (
                df_long["variable_mois"].astype(str)
                .str.replace(r"\s*20\d{2}-\d{2}$", "", regex=True)
                .str.strip()
            )
            df_long = df_long.rename(columns={code_col: "code_atc3", lib_col: "libelle_atc3"})
            df_long["annee"] = annee
            df_long["semestre"] = semestre
            df_long["fichier_source"] = file_path.name
            df_long = df_long[[
                "annee", "semestre", "mois", "code_atc3", "libelle_atc3",
                "indicateur", "valeur", "fichier_source"
            ]]
            frames.append(df_long)
        except Exception as e:
            load_errors.append({"fichier": file_path.name, "annee": annee, "semestre": semestre, "erreur": str(e)})

    if load_errors:
        print("ERREURS DE CHARGEMENT:")
        display(pd.DataFrame(load_errors))

    if not frames:
        raise ValueError("Aucun fichier Ameli mensuel charge")

    df_atc3_all = pd.concat(frames, ignore_index=True)
    df_atc3_all = df_atc3_all.dropna(subset=["mois"]).sort_values(["mois", "code_atc3", "indicateur"]).reset_index(drop=True)

    print("df_atc3_all reconstruit (avec 2014):", df_atc3_all.shape)
    print("Periode:", df_atc3_all["mois"].min(), "a", df_atc3_all["mois"].max())
    display(df_atc3_all.head(12))

else:
    print("Skipped optional OPEN_MEDIC cell: " + SKIP_REASON)


Skipped optional OPEN_MEDIC cell: OPEN_MEDIC source files are not available in sources/open_medic: 2025-01-a-06_medic-am-par-classe-atc_serie-mensuelle-labellisee.xlsx, OPEN_PHMEV_2024.CSV, 2017-a-2025_retroced-am_serie-annuelle..xlsx


In [17]:
if RUN_OPEN_MEDIC:
    # Point 1 — Audit des sources hospitalières (OPEN_PHMEV_2024 + Retroced'AM 2017-2025)
    from pathlib import Path
    import re
    import pandas as pd
    import unicodedata

    phmev_path = SOURCE_DIR / "OPEN_PHMEV_2024.CSV"
    retro_path = SOURCE_DIR / "2017-a-2025_retroced-am_serie-annuelle..xlsx"
    assert phmev_path.exists(), f"Fichier introuvable: {phmev_path}"
    assert retro_path.exists(), f"Fichier introuvable: {retro_path}"

    def _ascii_norm(text):
        s = unicodedata.normalize("NFKD", str(text)).encode("ascii", "ignore").decode("ascii")
        return re.sub(r"\s+", " ", s).strip()

    def _to_num_series(s):
        return pd.to_numeric(
            s.astype(str)
            .str.replace("\u00a0", "", regex=False)
            .str.replace(" ", "", regex=False)
            .str.replace(",", ".", regex=False),
            errors="coerce",
        ).fillna(0)

    print("Apercu OPEN_PHMEV_2024 colonnes:")
    phmev_cols = pd.read_csv(phmev_path, sep=";", nrows=0, encoding="latin-1").columns.tolist()
    print(phmev_cols)

    usecols = ["atc3", "L_ATC3", "region_etb", "BOITES", "REM"]
    r06_rows = 0
    sum_boites = 0.0
    sum_rem = 0.0
    region_boites = {}

    for chunk in pd.read_csv(
        phmev_path,
        sep=";",
        usecols=usecols,
        encoding="latin-1",
        low_memory=False,
        chunksize=300_000,
    ):
        chunk["atc3"] = chunk["atc3"].astype(str).str.strip().str.upper()
        sub = chunk.loc[chunk["atc3"].eq("R06A")].copy()
        if sub.empty:
            continue

        r06_rows += len(sub)
        sub["BOITES"] = _to_num_series(sub["BOITES"])
        sub["REM"] = _to_num_series(sub["REM"])
        sum_boites += sub["BOITES"].sum()
        sum_rem += sub["REM"].sum()

        for reg, val in sub.groupby("region_etb")["BOITES"].sum().items():
            region_boites[reg] = region_boites.get(reg, 0.0) + float(val)

    phmev_r06a_region = (
        pd.Series(region_boites, name="boites_r06a_2024")
        .rename_axis("region_etb")
        .reset_index()
        .sort_values("boites_r06a_2024", ascending=False)
        .reset_index(drop=True)
    )

    print(f"OPEN_PHMEV_2024 -> lignes R06A: {r06_rows:,}")
    print(f"OPEN_PHMEV_2024 -> boites R06A total: {sum_boites:,.0f}")
    print(f"OPEN_PHMEV_2024 -> remboursement R06A total: {sum_rem:,.0f}")
    display(phmev_r06a_region.head(10))

    # --- Retroced'AM ---
    retro_tmp = pd.read_excel(retro_path, sheet_name="Retroced'AM", header=None)

    header_idx = None
    for i in range(min(50, len(retro_tmp))):
        vals = [_ascii_norm(v).lower().replace(" ", "_") for v in retro_tmp.iloc[i].tolist()]
        if "code_atc" in vals:
            header_idx = i
            break

    if header_idx is None:
        preview = retro_tmp.head(12).iloc[:, :12]
        raise ValueError("Colonne code_atc introuvable dans Retroced'AM. Apercu:\n" + preview.to_string(index=False))

    print(f"Header Retroced'AM detecte a la ligne (0-based): {header_idx}")

    raw_cols = [_ascii_norm(v) for v in retro_tmp.iloc[header_idx].tolist()]
    raw_cols = [c if c else f"col_{j}" for j, c in enumerate(raw_cols)]

    # rendre les colonnes uniques
    seen = {}
    retro_cols = []
    for c in raw_cols:
        if c in seen:
            seen[c] += 1
            retro_cols.append(f"{c}_{seen[c]}")
        else:
            seen[c] = 0
            retro_cols.append(c)

    retro_raw = retro_tmp.iloc[header_idx + 1 :].copy()
    retro_raw.columns = retro_cols

    code_col = next((c for c in retro_raw.columns if c.lower().replace(" ", "_") == "code_atc"), None)
    if code_col is None:
        code_col = next((c for c in retro_raw.columns if "code_atc" in c.lower().replace(" ", "_")), None)
    if code_col is None:
        raise ValueError(f"Colonne code_atc introuvable apres detection. Colonnes: {list(retro_raw.columns)[:25]}")

    retro_r06 = retro_raw.loc[retro_raw[code_col].astype(str).str.startswith("R06", na=False)].copy()

    measure_cols = []
    for c in retro_r06.columns:
        c_norm = _ascii_norm(c)
        if re.match(r"^(Base|Remb|Unites)\s20\d{2}$", c_norm):
            measure_cols.append(c)

    rows = []
    for col in measure_cols:
        c_norm = _ascii_norm(col)
        m = re.match(r"^(Base|Remb|Unites)\s(20\d{2})$", c_norm)
        if not m:
            continue

        metric = m.group(1)
        year = int(m.group(2))
        val = _to_num_series(retro_r06[col]).sum()
        rows.append({"annee": year, "metrique": metric, "valeur": val})

    if rows:
        retro_r06a_annual = (
            pd.DataFrame(rows)
            .pivot(index="annee", columns="metrique", values="valeur")
            .reset_index()
            .sort_values("annee")
            .reset_index(drop=True)
        )
        display(retro_r06a_annual)
    else:
        print("Aucune colonne annuelle Base/Remb/Unites detectee pour R06 dans Retroced'AM.")

    print("Conclusion faisabilite:")
    print("- OPEN_PHMEV_2024: exploitable pour un zoom hospitalier R06A, mais limite a l annee 2024.")
    print("- Retroced'AM 2017-2025: exploitable en annualise (pas mensuel), utile pour contexte long terme hospitalier.")

else:
    print("Skipped optional OPEN_MEDIC cell: " + SKIP_REASON)


Skipped optional OPEN_MEDIC cell: OPEN_MEDIC source files are not available in sources/open_medic: 2025-01-a-06_medic-am-par-classe-atc_serie-mensuelle-labellisee.xlsx, OPEN_PHMEV_2024.CSV, 2017-a-2025_retroced-am_serie-annuelle..xlsx


In [18]:
if RUN_OPEN_MEDIC:
    # Point 2 — Analyse graphique antihistaminiques vs pollen/HPI (niveau national mensuel)
    import matplotlib.pyplot as plt
    import seaborn as sns

    if "df_atc3_all" not in locals():
        raise ValueError("df_atc3_all absent. Execute l Etape 7 avant.")

    def _norm_text(x):
        s = unicodedata.normalize("NFKD", str(x)).encode("ascii", "ignore").decode("ascii")
        return re.sub(r"\s+", " ", s).strip().lower()

    df_r06 = df_atc3_all.loc[df_atc3_all["code_atc3"].eq("R06A")].copy()
    ind_candidates = [
        ind for ind in df_r06["indicateur"].dropna().unique().tolist()
        if ("boite" in _norm_text(ind)) and ("rembours" in _norm_text(ind))
    ]
    if not ind_candidates:
        raise ValueError("Indicateur boites remboursées introuvable dans df_atc3_all pour R06A")
    target_indicator = sorted(ind_candidates)[0]

    r06_monthly_fr = (
        df_r06.loc[df_r06["indicateur"].eq(target_indicator), ["mois", "valeur"]]
        .assign(boites_r06a=lambda d: pd.to_numeric(d["valeur"], errors="coerce"))
        .groupby("mois", as_index=False)["boites_r06a"].sum()
        .dropna()
    )

    pollen_daily = pd.read_csv("Data/processed/pollen_weather_merged.csv", parse_dates=["date"])
    hpi_daily = pd.read_csv("Data/processed/hpi_daily.csv", parse_dates=["date"])
    pollen_cols = ["birch_pollen", "alder_pollen", "grass_pollen", "olive_pollen", "mugwort_pollen", "ragweed_pollen"]
    pollen_daily["pollen_total"] = pollen_daily[pollen_cols].sum(axis=1, min_count=1)

    daily_fr = pollen_daily[["date", "city", "pollen_total"]].merge(
        hpi_daily[["date", "city", "HPI"]],
        on=["date", "city"],
        how="inner",
    )
    daily_fr = daily_fr.groupby("date", as_index=False).agg(pollen_total=("pollen_total", "mean"), hpi=("HPI", "mean"))
    daily_fr["mois"] = daily_fr["date"].dt.to_period("M").astype(str)
    climate_monthly_fr = daily_fr.groupby("mois", as_index=False).agg(pollen_moyen=("pollen_total", "mean"), hpi_moyen=("hpi", "mean"))

    analysis_fr = climate_monthly_fr.merge(r06_monthly_fr, on="mois", how="inner")
    analysis_fr = analysis_fr.sort_values("mois").reset_index(drop=True)
    analysis_fr["date_mois"] = pd.to_datetime(analysis_fr["mois"] + "-01")

    print("Periode analysee (merge climat + R06A):", analysis_fr["mois"].min(), "a", analysis_fr["mois"].max())
    display(analysis_fr.head(12))

    fig, ax1 = plt.subplots(figsize=(14, 5))
    ax1.plot(analysis_fr["date_mois"], analysis_fr["boites_r06a"], color="#d1495b", linewidth=2.2, label="R06A boites")
    ax1.set_ylabel("Boites R06A", color="#d1495b")
    ax1.tick_params(axis="y", labelcolor="#d1495b")

    ax2 = ax1.twinx()
    ax2.plot(analysis_fr["date_mois"], analysis_fr["pollen_moyen"], color="#00798c", linewidth=1.8, alpha=0.9, label="Pollen moyen")
    ax2.plot(analysis_fr["date_mois"], analysis_fr["hpi_moyen"], color="#edae49", linewidth=1.8, alpha=0.9, label="HPI moyen")
    ax2.set_ylabel("Pollen / HPI", color="#2f2f2f")

    lines = ax1.get_lines() + ax2.get_lines()
    labels = [l.get_label() for l in lines]
    ax1.legend(lines, labels, loc="upper left")
    ax1.set_title("Antihistaminiques R06A vs pollen et HPI (mensuel, France)")
    ax1.set_xlabel("Mois")
    plt.tight_layout()
    plt.show()

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
    sns.regplot(data=analysis_fr, x="pollen_moyen", y="boites_r06a", ax=axes[0], scatter_kws={"alpha": 0.7, "s": 42}, line_kws={"color": "#d1495b"})
    axes[0].set_title("R06A vs pollen moyen")
    axes[0].set_xlabel("Pollen moyen mensuel")
    axes[0].set_ylabel("Boites R06A")

    sns.regplot(data=analysis_fr, x="hpi_moyen", y="boites_r06a", ax=axes[1], scatter_kws={"alpha": 0.7, "s": 42}, line_kws={"color": "#00798c"})
    axes[1].set_title("R06A vs HPI moyen")
    axes[1].set_xlabel("HPI moyen mensuel")
    axes[1].set_ylabel("Boites R06A")
    plt.tight_layout()
    plt.show()

else:
    print("Skipped optional OPEN_MEDIC cell: " + SKIP_REASON)


Skipped optional OPEN_MEDIC cell: OPEN_MEDIC source files are not available in sources/open_medic: 2025-01-a-06_medic-am-par-classe-atc_serie-mensuelle-labellisee.xlsx, OPEN_PHMEV_2024.CSV, 2017-a-2025_retroced-am_serie-annuelle..xlsx


In [19]:
if RUN_OPEN_MEDIC:
    # Point 2 bis — Effet "haut pollen / haut HPI" sur les remboursements antihistaminiques
    if "analysis_fr" not in locals():
        raise ValueError("analysis_fr absent. Execute le Point 2 avant.")

    q_pollen = analysis_fr["pollen_moyen"].quantile(0.75)
    q_hpi = analysis_fr["hpi_moyen"].quantile(0.75)

    analysis_fr = analysis_fr.copy()
    analysis_fr["pollen_haut"] = analysis_fr["pollen_moyen"] >= q_pollen
    analysis_fr["hpi_haut"] = analysis_fr["hpi_moyen"] >= q_hpi
    analysis_fr["regime_expo"] = "Autres mois"
    analysis_fr.loc[analysis_fr["pollen_haut"], "regime_expo"] = "Pollen haut"
    analysis_fr.loc[analysis_fr["hpi_haut"], "regime_expo"] = "HPI haut"
    analysis_fr.loc[analysis_fr["pollen_haut"] & analysis_fr["hpi_haut"], "regime_expo"] = "Pollen haut + HPI haut"

    summary_expo = (
        analysis_fr.groupby("regime_expo", as_index=False)
        .agg(
            n_mois=("mois", "count"),
            boites_moyennes=("boites_r06a", "mean"),
            pollen_moyen=("pollen_moyen", "mean"),
            hpi_moyen=("hpi_moyen", "mean"),
        )
    )
    display(summary_expo.sort_values("boites_moyennes", ascending=False))

    order = ["Autres mois", "Pollen haut", "HPI haut", "Pollen haut + HPI haut"]
    plt.figure(figsize=(10, 4.5))
    sns.boxplot(data=analysis_fr, x="regime_expo", y="boites_r06a", order=order)
    plt.title("Distribution des boites R06A selon le niveau d exposition pollen/HPI")
    plt.xlabel("Regime d exposition")
    plt.ylabel("Boites R06A")
    plt.xticks(rotation=15)
    plt.tight_layout()
    plt.show()

else:
    print("Skipped optional OPEN_MEDIC cell: " + SKIP_REASON)


Skipped optional OPEN_MEDIC cell: OPEN_MEDIC source files are not available in sources/open_medic: 2025-01-a-06_medic-am-par-classe-atc_serie-mensuelle-labellisee.xlsx, OPEN_PHMEV_2024.CSV, 2017-a-2025_retroced-am_serie-annuelle..xlsx


In [20]:
if RUN_OPEN_MEDIC:
    # Point 3 — Comparaison 2014 vs 2022-2024 (profil saisonnier antihistaminiques)
    if "df_atc3_all" not in locals():
        raise ValueError("df_atc3_all absent. Execute l Etape 7 avant.")

    def _norm_text2(x):
        s = unicodedata.normalize("NFKD", str(x)).encode("ascii", "ignore").decode("ascii")
        return re.sub(r"\s+", " ", s).strip().lower()

    df_r06 = df_atc3_all.loc[df_atc3_all["code_atc3"].eq("R06A")].copy()
    ind_candidates = [
        ind for ind in df_r06["indicateur"].dropna().unique().tolist()
        if ("boite" in _norm_text2(ind)) and ("rembours" in _norm_text2(ind))
    ]
    target_indicator = sorted(ind_candidates)[0]

    r06_monthly = (
        df_r06.loc[df_r06["indicateur"].eq(target_indicator), ["mois", "valeur"]]
        .assign(boites_r06a=lambda d: pd.to_numeric(d["valeur"], errors="coerce"))
        .groupby("mois", as_index=False)["boites_r06a"].sum()
        .dropna()
    )
    r06_monthly["annee"] = r06_monthly["mois"].str[:4].astype(int)
    r06_monthly["mois_num"] = r06_monthly["mois"].str[-2:].astype(int)

    compare_years = [2014, 2022, 2023, 2024]
    r06_compare = r06_monthly.loc[r06_monthly["annee"].isin(compare_years)].copy()

    season_profile = (
        r06_compare.groupby(["annee", "mois_num"], as_index=False)["boites_r06a"].mean()
    )
    display(season_profile.head(20))

    plt.figure(figsize=(12.5, 5))
    sns.lineplot(data=season_profile, x="mois_num", y="boites_r06a", hue="annee", marker="o", palette="tab10")
    plt.title("Comparaison mensuelle des remboursements R06A: 2014 vs 2022-2024")
    plt.xlabel("Mois")
    plt.ylabel("Boites R06A")
    plt.xticks(range(1, 13))
    plt.tight_layout()
    plt.show()

    year_summary = (
        r06_compare.groupby("annee", as_index=False)
        .agg(total_boites=("boites_r06a", "sum"), moyenne_mensuelle=("boites_r06a", "mean"))
        .sort_values("annee")
    )
    display(year_summary)

else:
    print("Skipped optional OPEN_MEDIC cell: " + SKIP_REASON)


Skipped optional OPEN_MEDIC cell: OPEN_MEDIC source files are not available in sources/open_medic: 2025-01-a-06_medic-am-par-classe-atc_serie-mensuelle-labellisee.xlsx, OPEN_PHMEV_2024.CSV, 2017-a-2025_retroced-am_serie-annuelle..xlsx


## Synthèse professionnelle — Pollen, HPI et remboursements d’antihistaminiques (R06A)

### 1) Objectif
Ce notebook vise à tester si la dynamique du pollen et du **Heat-Pollution Index (HPI)** est associée aux remboursements mensuels d’antihistaminiques (`ATC3 = R06A`), puis à comparer l’apport explicatif de deux modèles :
- **Modèle A** : pollen seul  
- **Modèle B** : pollen + HPI

### 2) Données mobilisées
- **Ameli mensuel (ATC3)** : `df_atc3_all` reconstruit sur **2014–2025** (fichiers semestriels, format long).
- **Climat / exposition** : pollen et HPI quotidiens agrégés au mois (2022–2024).
- **Hospitalier** :
- `OPEN_PHMEV_2024.CSV` (exploitable pour R06A, niveau établissement/région).
- `2017-a-2025_retroced-am_serie-annuelle..xlsx` (annualisé, non contributif sur R06 dans l’état actuel).

### 3) Résultats principaux

#### 3.1 Association globale pollen / ventes R06A
- Co-saisonnalité nette : les périodes de pollen élevé coïncident avec des volumes plus élevés de boîtes R06A.
- Sur 2022–2024, l’analyse mensuelle montre un signal robuste entre pollen et remboursements.

#### 3.2 Apport du HPI (test central A vs B)
- **Modèle A (pollen)** : `R² = 0.543`
- **Modèle B (pollen + HPI)** : `R² = 0.546`
- **Gain** : `ΔR² = +0.0025`
- **F-test imbriqué** : `p = 0.671` (non significatif)

Conclusion : dans ce setup, **le HPI n’apporte pas d’amélioration statistiquement significative** au-delà du pollen seul.

#### 3.3 Analyse par ville
- Strasbourg : `ΔR² = +0.053`, `p = 0.059` (proche du seuil, non significatif)
- Paris : `ΔR² = +0.045`, `p = 0.135` (non significatif)
- Bordeaux : `ΔR² = +0.006`, `p = 0.529` (non significatif)
- Marseille : `ΔR² = +0.002`, `p = 0.706` (non significatif)

Conclusion : pas de preuve statistique solide d’un gain du modèle B par ville dans l’état actuel.

#### 3.4 Segmentation “forte exposition”
- `Pollen haut + HPI haut` : **5.11M** boîtes/mois
- `Pollen haut` : **4.85M**
- `Autres mois` : **3.94M**
- `HPI haut` seul : **3.93M**

Lecture : l’élévation est surtout portée par les mois de **pollen élevé**.

#### 3.5 Perspective temporelle 2014 vs 2022–2024
- Total annuel R06A :
- 2014 : **44.4M**
- 2022 : **48.8M**
- 2023 : **49.4M**
- 2024 : **52.8M**
- Progression 2014 → 2024 : **~+18.8%**, avec saisonnalité printanière persistante.

#### 3.6 Focus hospitalier (2024)
- `OPEN_PHMEV_2024` (R06A) :
- **81 587** lignes
- **4 042 269** boîtes
- **4 538 344** remboursés
- Région la plus contributrice : **11 (Île-de-France)** avec **1 005 359** boîtes (~25%).

### 4) Limites méthodologiques
- La cible “ville” est estimée par allocation populationnelle (pas une observation mensuelle territoriale directe).
- Le HPI peut partager une saisonnalité commune avec le pollen, limitant son gain marginal en régression linéaire simple.
- Les sources hospitalières ne sont pas alignées temporellement au même niveau de granularité que le pipeline principal mensuel.

### 5) Conclusion exécutive
- Le notebook montre une **association claire entre hausse du pollen et hausse des remboursements R06A**.
- En revanche, il ne démontre pas, à ce stade, une **valeur ajoutée statistiquement significative du HPI** par rapport au pollen seul.
- Le volet hospitalier confirme l’importance de R06A en 2024 et peut enrichir la discussion, mais pas encore renforcer une preuve causale sur la dynamique mensuelle.
